In [ ]:
!pip install groq python-whois requests beautifulsoup4 dnspython

In [ ]:
import os
import json
import time
import re
import requests
import whois
import dns.resolver
from datetime import datetime, timezone
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from groq import Groq
from google.colab import userdata

# Load API keys securely from Colab Secrets
os.environ["GROQ_API_KEY"]       = userdata.get("GROQ_API_KEY")
os.environ["VIRUSTOTAL_API_KEY"] = userdata.get("VIRUSTOTAL_API_KEY")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
VT_KEY = os.environ["VIRUSTOTAL_API_KEY"]

MODEL = "llama-3.3-70b-versatile"

print("✅ Libraries loaded")
print("✅ Credentials loaded")

✅ Libraries loaded
✅ Credentials loaded


In [ ]:
def whois_lookup(domain: str) -> dict:
    """
    Looks up WHOIS registration data for a domain.
    Returns domain age, registrar, country, and risk signals.

    CONCEPT: We extract 3 key signals:
    1. Domain age in days (newer = more suspicious)
    2. Registrant country (should match claimed company location)
    3. Expiry period (1 year only = suspicious)
    """
    print(f"\n  🔍 [TOOL] WHOIS lookup: {domain}")

    try:
        w = whois.whois(domain)

        # Extract creation date — handle both single date and list
        creation = w.creation_date
        if isinstance(creation, list):
            creation = creation[0]

        # Extract expiry date
        expiry = w.expiration_date
        if isinstance(expiry, list):
            expiry = expiry[0]

        # Calculate domain age in days
        if creation:
            # Make creation date timezone-aware if it isn't
            if creation.tzinfo is None:
                creation = creation.replace(tzinfo=timezone.utc)
            now = datetime.now(timezone.utc)
            age_days = (now - creation).days
        else:
            age_days = None

        # Determine risk level from age
        if age_days is None:
            age_risk = "unknown"
        elif age_days < 30:
            age_risk = "VERY HIGH — domain less than 30 days old"
        elif age_days < 180:
            age_risk = "HIGH — domain less than 6 months old"
        elif age_days < 365:
            age_risk = "MEDIUM — domain less than 1 year old"
        else:
            age_risk = "LOW — domain is established"

        result = {
            "domain"          : domain,
            "creation_date"   : str(creation) if creation else "not found",
            "expiry_date"     : str(expiry) if expiry else "not found",
            "age_days"        : age_days,
            "age_risk"        : age_risk,
            "registrar"       : str(w.registrar) if w.registrar else "not found",
            "registrant_country": str(w.country) if w.country else "not found",
            "name_servers"    : list(w.name_servers)[:3] if w.name_servers else [],
            "status"          : "success"
        }

        print(f"     Domain age  : {age_days} days")
        print(f"     Risk level  : {age_risk}")
        print(f"     Country     : {result['registrant_country']}")
        print(f"     Registrar   : {result['registrar']}")

        return result

    except Exception as e:
        print(f"     ⚠ WHOIS lookup failed: {e}")
        return {
            "domain": domain,
            "status": "failed",
            "error" : str(e),
            "age_risk": "unknown — lookup failed"
        }


In [ ]:
def virustotal_scan(url: str) -> dict:
    """
    Checks a URL against VirusTotal's 70+ security engines.
    Returns how many engines flagged it and their verdicts.

    CONCEPT: The VirusTotal API has two steps:
    Step 1 — Submit the URL → get a scan ID
    Step 2 — Query the scan ID → get results
    We wait a few seconds between steps for analysis to complete.
    """
    print(f"\n  🔍 [TOOL] VirusTotal scan: {url}")

    headers = {"x-apikey": VT_KEY}

    try:
        # Step 1: Submit URL for scanning
        submit_response = requests.post(
            "https://www.virustotal.com/api/v3/urls",
            headers=headers,
            data={"url": url},
            timeout=15
        )

        if submit_response.status_code != 200:
            return {
                "url"   : url,
                "status": "failed",
                "error" : f"Submission failed: {submit_response.status_code}"
            }

        # Extract the analysis ID from the response
        analysis_id = submit_response.json()["data"]["id"]
        print(f"     Scan submitted. Waiting for results...")

        # Step 2: Wait then fetch results
        time.sleep(5)   # Give VirusTotal time to analyze

        result_response = requests.get(
            f"https://www.virustotal.com/api/v3/analyses/{analysis_id}",
            headers=headers,
            timeout=15
        )

        if result_response.status_code != 200:
            return {
                "url"   : url,
                "status": "failed",
                "error" : f"Result fetch failed: {result_response.status_code}"
            }

        stats = result_response.json()["data"]["attributes"]["stats"]

        malicious  = stats.get("malicious", 0)
        suspicious = stats.get("suspicious", 0)
        harmless   = stats.get("harmless", 0)
        undetected = stats.get("undetected", 0)
        total      = malicious + suspicious + harmless + undetected

        # Determine risk from engine verdicts
        if malicious >= 5:
            vt_risk = "VERY HIGH — flagged by multiple security engines"
        elif malicious >= 2:
            vt_risk = "HIGH — flagged by some security engines"
        elif malicious >= 1 or suspicious >= 3:
            vt_risk = "MEDIUM — flagged by at least one engine"
        else:
            vt_risk = "LOW — not flagged by security engines"

        result = {
            "url"           : url,
            "malicious"     : malicious,
            "suspicious"    : suspicious,
            "harmless"      : harmless,
            "total_engines" : total,
            "vt_risk"       : vt_risk,
            "status"        : "success"
        }

        print(f"     Malicious engines : {malicious}/{total}")
        print(f"     Suspicious engines: {suspicious}/{total}")
        print(f"     Risk level        : {vt_risk}")

        return result

    except Exception as e:
        print(f"     ⚠ VirusTotal scan failed: {e}")
        return {
            "url"   : url,
            "status": "failed",
            "error" : str(e)
        }


In [ ]:
def scrape_website(url: str) -> dict:
    """
    Visits a suspicious URL and extracts content for analysis.
    Uses BeautifulSoup to find red flags in the page content.

    YOUR SKILL: This is exactly the BeautifulSoup + requests
    pattern you already know from web scraping.
    """
    print(f"\n  🔍 [TOOL] Scraping website: {url}")

    # Mimic a real browser to avoid bot detection
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        # Extract page title
        title = soup.title.string.strip() if soup.title else "No title found"

        # Extract all visible text (cleaned)
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        text = " ".join(soup.get_text().split())[:2000]  # first 2000 chars

        # Look for suspicious patterns in the page
        text_lower = text.lower()

        scam_patterns = {
            "login_form"      : bool(soup.find("form")),
            "password_field"  : bool(soup.find("input", {"type": "password"})),
            "urgency_language": any(w in text_lower for w in [
                "urgent", "expires", "limited time", "act now",
                "within 24 hours", "immediately", "do not delay"
            ]),
            "prize_language"  : any(w in text_lower for w in [
                "congratulations", "winner", "prize", "reward",
                "you have won", "lucky draw", "selected"
            ]),
            "financial_request": any(w in text_lower for w in [
                "bank account", "wire transfer", "processing fee",
                "advance payment", "western union", "moneygram"
            ]),
            "credential_request": any(w in text_lower for w in [
                "enter your password", "verify your account",
                "confirm your details", "update your information"
            ]),
        }

        flags_found = [k for k, v in scam_patterns.items() if v]

        result = {
            "url"            : url,
            "page_title"     : title,
            "status_code"    : response.status_code,
            "scam_patterns"  : scam_patterns,
            "flags_found"    : flags_found,
            "flags_count"    : len(flags_found),
            "page_text_sample": text[:500],
            "status"         : "success"
        }

        print(f"     Page title   : {title[:60]}")
        print(f"     Flags found  : {flags_found}")

        return result

    except requests.exceptions.ConnectionError:
        print(f"     ⚠ Website unreachable — domain may not exist")
        return {
            "url"   : url,
            "status": "unreachable",
            "note"  : "Domain exists in email but website does not load — strong scam signal"
        }
    except Exception as e:
        print(f"     ⚠ Scraping failed: {e}")
        return {"url": url, "status": "failed", "error": str(e)}


In [ ]:
def analyze_email_domain(email: str) -> dict:
    """
    Analyzes an email address for scam signals.
    Checks domain type, MX records, and prefix patterns.

    CONCEPT: dns.resolver queries real DNS servers to check
    if the domain is set up to send/receive email legitimately.
    """
    print(f"\n  🔍 [TOOL] Email domain analysis: {email}")

    try:
        # Extract domain from email
        if "@" not in email:
            return {"email": email, "status": "invalid", "note": "Not a valid email address"}

        prefix, domain = email.split("@", 1)

        # Check if it is a free email provider
        free_providers = [
            "gmail.com", "yahoo.com", "hotmail.com", "outlook.com",
            "live.com", "protonmail.com", "icloud.com", "aol.com",
            "yandex.com", "mail.com"
        ]
        is_free_provider = domain.lower() in free_providers

        # Check for suspicious prefix patterns
        suspicious_prefixes = [
            "hralert", "hr-alert", "hr_alert", "noreply-hr",
            "jobs-alert", "alert-team", "recruitment-alert",
            "info-team", "support-team", "admin-alert"
        ]
        suspicious_prefix = any(p in prefix.lower() for p in suspicious_prefixes)

        # Check MX records — does this domain actually handle email?
        try:
            mx_records = dns.resolver.resolve(domain, "MX")
            has_mx     = True
            mx_list    = [str(r.exchange) for r in mx_records][:3]
        except Exception:
            has_mx  = False
            mx_list = []

        # Determine overall email risk
        risk_signals = []
        if is_free_provider:
            risk_signals.append("free email provider used for business communication")
        if suspicious_prefix:
            risk_signals.append(f"suspicious prefix pattern: {prefix}")
        if not has_mx:
            risk_signals.append("domain has no MX records — cannot legitimately send email")

        if len(risk_signals) >= 2:
            email_risk = "HIGH"
        elif len(risk_signals) == 1:
            email_risk = "MEDIUM"
        else:
            email_risk = "LOW"

        result = {
            "email"             : email,
            "prefix"            : prefix,
            "domain"            : domain,
            "is_free_provider"  : is_free_provider,
            "suspicious_prefix" : suspicious_prefix,
            "has_mx_records"    : has_mx,
            "mx_records"        : mx_list,
            "risk_signals"      : risk_signals,
            "email_risk"        : email_risk,
            "status"            : "success"
        }

        print(f"     Domain       : {domain}")
        print(f"     Free provider: {is_free_provider}")
        print(f"     Suspicious prefix: {suspicious_prefix}")
        print(f"     Has MX records: {has_mx}")
        print(f"     Email risk   : {email_risk}")

        return result

    except Exception as e:
        print(f"     ⚠ Email analysis failed: {e}")
        return {"email": email, "status": "failed", "error": str(e)}



In [ ]:
def check_domain_mismatch(sender_email: str, body_domains: list) -> dict:
    """
    Checks whether the sender domain matches domains mentioned
    in the email body. A mismatch is a strong scam signal.

    CONCEPT: Legitimate companies always send from the same
    domain their website and brand runs on. Scammers separate
    these deliberately.
    """
    print(f"\n  🔍 [TOOL] Domain mismatch check")

    if "@" not in sender_email:
        return {"status": "invalid", "note": "No valid sender email"}

    sender_domain = sender_email.split("@")[1].lower()
    body_domains  = [d.lower().strip() for d in body_domains]

    # Check if sender domain appears in body domains
    exact_match = sender_domain in body_domains

    # Check for partial brand match
    # e.g. sender: hr@wadi-group.com  body: wadigroup.com
    # These look related but are different domains
    brand_matches = [
        d for d in body_domains
        if any(part in d for part in sender_domain.split(".")[:-1])
    ]

    mismatches = [d for d in body_domains if d != sender_domain]

    if not exact_match and mismatches:
        mismatch_risk = "HIGH — sender domain does not match body domains"
    elif brand_matches and not exact_match:
        mismatch_risk = "MEDIUM — similar but not identical domains found"
    else:
        mismatch_risk = "LOW — domains are consistent"

    result = {
        "sender_domain" : sender_domain,
        "body_domains"  : body_domains,
        "exact_match"   : exact_match,
        "mismatches"    : mismatches,
        "mismatch_risk" : mismatch_risk,
        "status"        : "success"
    }

    print(f"     Sender domain : {sender_domain}")
    print(f"     Body domains  : {body_domains}")
    print(f"     Match found   : {exact_match}")
    print(f"     Mismatch risk : {mismatch_risk}")

    return result

In [ ]:
# CONCEPT: This is the most important new concept in Phase 2.
# We must tell the LLM what tools exist and how to call them.
# We do this using a "tool schema" — a JSON description of:
#   - The tool's name
#   - What it does
#   - What parameters it needs
#   - What type each parameter is
#
# The LLM reads these schemas and decides when to call each tool.
# It does NOT call Python directly — it returns a structured
# message saying "I want to call THIS tool with THESE arguments"
# and then WE run the Python function and return the result.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "whois_lookup",
            "description": (
                "Look up WHOIS registration data for a domain name. "
                "Use this when you find a domain or email address in the message. "
                "Returns domain age, registrar, country, and risk level. "
                "Newly registered domains are a strong scam signal."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {
                        "type": "string",
                        "description": "The domain name to look up, e.g. wadialsagroup.com"
                    }
                },
                "required": ["domain"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "virustotal_scan",
            "description": (
                "Scan a URL against VirusTotal's 70+ security engines. "
                "Use this when the message contains a clickable URL or link. "
                "Returns how many security engines flagged the URL as malicious."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "The full URL to scan, e.g. https://suspicious-site.com/login"
                    }
                },
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "scrape_website",
            "description": (
                "Visit a website and extract its content to look for scam patterns. "
                "Use this when you want to inspect what a suspicious URL actually shows. "
                "Looks for login forms, urgency language, prize claims, and financial requests."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "The full URL to visit and scrape"
                    }
                },
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_email_domain",
            "description": (
                "Analyze an email address for scam signals. "
                "Use this when the message contains a sender email address. "
                "Checks if it is a free provider, suspicious prefix pattern, "
                "and whether the domain has valid MX records."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "The full email address to analyze, e.g. hralert@wadialsagroup.com"
                    }
                },
                "required": ["email"]
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "check_domain_mismatch",
        "description": (
            "Check whether the sender email domain matches domains "
            "mentioned in the email body. Use this when you find both "
            "a sender email address AND domain names or URLs in the "
            "message body. A mismatch is a strong scam indicator called "
            "a domain mismatch attack."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "sender_email": {
                    "type": "string",
                    "description": "The sender email address e.g. hr@company.com"
                },
                "body_domains": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of domains found in the message body"
                }
            },
            "required": ["sender_email", "body_domains"]
        }
    }
}
]


In [ ]:
# CONCEPT: When the LLM decides to call a tool, it returns the
# tool name and arguments. We need a dispatcher that maps the
# tool name to the actual Python function and runs it.
# Think of this as the "hands" that execute what the LLM decided.

def dispatch_tool(tool_name: str, tool_args: dict) -> str:
    """
    Receives a tool call decision from the LLM,
    runs the corresponding Python function,
    and returns the result as a JSON string.

    CONCEPT: The result must be returned as a string because
    the LLM only speaks in text. We JSON-encode the dict result
    so the LLM can read and reason over the evidence.
    """
    tool_map = {
        "whois_lookup"        : whois_lookup,
        "virustotal_scan"     : virustotal_scan,
        "scrape_website"      : scrape_website,
        "analyze_email_domain": analyze_email_domain,
        "check_domain_mismatch": check_domain_mismatch,
    }

    if tool_name not in tool_map:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})

    # Call the actual Python function with the LLM's chosen arguments
    result = tool_map[tool_name](**tool_args)

    # Return as JSON string so the LLM can read it
    return json.dumps(result, default=str)



In [ ]:
# CONCEPT: The system prompt now tells the agent about its tools
# and instructs it to ALWAYS use them before forming a verdict.
# Notice how we guide the agent's investigation strategy.

SYSTEM_PROMPT = """
You are an expert fraud analyst with 15 years of experience.
You have access to powerful investigation tools.

## YOUR INVESTIGATION STRATEGY:
1. Read the message carefully
2. Extract ALL email addresses, domains, and URLs present
3. Call analyze_email_domain for every email address found
4. Call whois_lookup for every domain found
5. Call virustotal_scan for every URL found
6. Call scrape_website if you want to inspect a URL's content
7. Only form your final verdict AFTER reviewing all tool results

## CRITICAL RULES:
- NEVER give a verdict before using your tools
- Tool results are FACTS — they override text-based impressions
- A domain registered less than 180 days ago is always suspicious
- A professional-sounding email does NOT make a scam legitimate
- Real UAE companies use .ae domains or long-established .com domains

## DOMAIN MISMATCH DETECTION:
If you find multiple domains in a message (sender domain vs domains
in the email body), always check ALL of them with whois_lookup.
A mismatch where:
- The sender domain is newly registered AND
- The body domain is older
is a strong indicator of a domain mismatch attack — a deliberate
technique to fool domain age checkers.
Always flag when sender domain ≠ body domain.

## FINAL VERDICT FORMAT:
After all tool calls, respond with ONLY this JSON structure:

{
  "is_scam": true or false,
  "confidence": 0 to 100,
  "scam_type": one of ["phishing", "lottery_fraud", "advance_fee",
               "romance_scam", "investment_fraud", "impersonation",
               "job_scam", "tech_support_scam", "not_a_scam", "unknown"],
  "evidence": {
    "whois_findings"  : "summary of domain age and registration details",
    "virustotal"      : "summary of URL scan results or not applicable",
    "email_analysis"  : "summary of email domain findings",
    "website_content" : "summary of scraped content or not applicable"
  },
  "red_flags"         : ["list", "of", "specific", "red", "flags"],
  "safe_flags"        : ["list", "of", "safe", "indicators"],
  "explanation"       : "Clear 2-3 sentence explanation for a non-technical person",
  "recommended_action": "What the recipient should do"
}
"""



In [ ]:
# CONCEPT: This is the most important function in Phase 2.
# It implements the agent loop:
#
#   1. Send message to LLM
#   2. LLM responds with tool calls OR a final answer
#   3. If tool calls → execute them → add results to conversation
#   4. Send updated conversation back to LLM
#   5. Repeat until LLM gives final answer (no more tool calls)
#
# This loop is what makes the agent "agentic" — it keeps going
# until IT decides it has enough evidence. We do not control
# how many tools it calls or in what order. The LLM decides.

def run_agent(message: str) -> dict:
    """
    Runs the full agentic investigation loop on a message.
    Returns structured verdict with real evidence.
    """
    print(f"\n{'='*60}")
    print(f"  PHASE 2 AGENT — STARTING INVESTIGATION")
    print(f"{'='*60}")
    print(f"Message preview: {message[:120]}...")
    print(f"{'='*60}")

    # Start conversation with system prompt + user message
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Investigate this message:\n\n{message}"}
    ]

    tool_call_count = 0
    max_iterations  = 10   # Safety limit — prevent infinite loops

    # ── THE AGENTIC LOOP ──────────────────────────────────────
    for iteration in range(max_iterations):

        print(f"\n  [Iteration {iteration + 1}] Calling LLM...")

        response = client.chat.completions.create(
            model      = MODEL,
            messages   = messages,
            tools      = TOOLS,          # Give LLM the tool schemas
            tool_choice= "auto",         # LLM decides when to use tools
            temperature= 0.1,
            max_tokens = 2000,
        )

        response_message = response.choices[0].message

        # CASE 1: LLM wants to call tools
        if response_message.tool_calls:

            # Add the LLM's tool call request to conversation history
            messages.append({
                "role"      : "assistant",
                "content"   : response_message.content,
                "tool_calls": [
                    {
                        "id"      : tc.id,
                        "type"    : "function",
                        "function": {
                            "name"     : tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    }
                    for tc in response_message.tool_calls
                ]
            })

            # Execute each tool the LLM requested
            for tool_call in response_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                tool_call_count += 1

                print(f"\n  → LLM requested: {tool_name}({tool_args})")

                # Run the actual Python tool
                tool_result = dispatch_tool(tool_name, tool_args)

                # Add tool result back to conversation
                # CONCEPT: The LLM needs to see what the tool returned
                # so it can reason over the evidence in the next iteration
                messages.append({
                    "role"        : "tool",
                    "tool_call_id": tool_call.id,
                    "content"     : tool_result
                })

                # Respect VirusTotal free tier rate limit
                if tool_name == "virustotal_scan":
                    print("     (Waiting 15s for VirusTotal rate limit...)")
                    time.sleep(15)

        # CASE 2: LLM has enough evidence and gives final verdict
        else:
            print(f"\n  ✅ Agent finished after {tool_call_count} tool calls")
            print(f"{'='*60}")

            # Parse the JSON verdict from the LLM's final response
            raw = response_message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            raw = raw.strip()

            try:
                verdict = json.loads(raw)
                verdict["tool_calls_made"] = tool_call_count
                return verdict
            except json.JSONDecodeError:
                return {
                    "is_scam"        : None,
                    "confidence"     : 0,
                    "explanation"    : "Could not parse final verdict",
                    "raw_response"   : raw,
                    "tool_calls_made": tool_call_count
                }

    # If we hit max iterations without a verdict
    return {
        "is_scam"        : None,
        "confidence"     : 0,
        "explanation"    : "Agent reached maximum iterations without verdict",
        "tool_calls_made": tool_call_count
    }



In [ ]:
# Updated display that shows the evidence collected by tools

def display_verdict(result: dict):
    """Displays the agent's verdict including tool evidence."""

    verdict    = "🚨 SCAM DETECTED" if result.get("is_scam") else "✅ LIKELY SAFE"
    confidence = result.get("confidence", 0)
    filled     = int(confidence / 10)
    bar        = "█" * filled + "░" * (10 - filled)

    print(f"\n{'─'*60}")
    print(f"  VERDICT : {verdict}")
    print(f"  Confidence : [{bar}] {confidence}%")
    print(f"  Scam type  : {result.get('scam_type','N/A').replace('_',' ').title()}")
    print(f"  Tools used : {result.get('tool_calls_made', 0)} tool calls made")
    print(f"{'─'*60}")

    # Show evidence collected by tools
    evidence = result.get("evidence", {})
    if evidence:
        print(f"\n  🔬 EVIDENCE COLLECTED:")
        for key, value in evidence.items():
            if value and value != "not applicable":
                label = key.replace("_", " ").title()
                print(f"     {label}: {value}")

    if result.get("red_flags"):
        print(f"\n  🚩 Red Flags:")
        for flag in result["red_flags"]:
            print(f"     • {flag}")

    if result.get("safe_flags"):
        print(f"\n  ✔  Safe Indicators:")
        for flag in result["safe_flags"]:
            print(f"     • {flag}")

    print(f"\n  📋 Explanation:")
    print(f"     {result.get('explanation','')}")

    print(f"\n  💡 Recommended Action:")
    print(f"     {result.get('recommended_action','')}")
    print(f"{'─'*60}\n")



In [ ]:
# We test the same scam email from Phase 1 that fooled the agent.
# Now watch it get caught with REAL evidence.

test_messages = [

    # TEST 1: The exact scam email that fooled Phase 1
    # Expected: Agent calls analyze_email_domain + whois_lookup
    # and catches it with real domain registration data
    """
    From: hralert@wadialsagroup.com
    Subject: Interview Invitation

    Dear Candidate,
    We are pleased to invite you for an interview at Wadi Al Salam Group.
    Your interview is scheduled for Monday at 10am.
    Please confirm your attendance by replying to this email.
    Bring your Emirates ID and bank account details for payroll setup.
    HR Department, Wadi Al Salam Group
    """,

    # TEST 2: Message with a suspicious URL
    # Expected: Agent calls virustotal_scan + scrape_website
    """
    Dear Emirates NBD Customer,
    Your account requires immediate verification.
    Click here: http://emiratesnbd-secure-login.tk/verify
    Failure to verify within 2 hours will suspend your account.
    """,

    # TEST 3: Legitimate message — should stay safe
    # Expected: Agent finds no suspicious signals
    """
    Hi, this is Ahmed from Careem support.
    Your recent ride on March 15 has been refunded to your wallet.
    You can check your balance in the Careem app.
    Ref: CR-2024-887234
    """,
]


In [ ]:
print("\n" + "="*60)
print("  ANTI-SCAM AGENT — PHASE 2 WITH TOOL CALLING")
print("  Real evidence. Not guesses.")
print("="*60)

results = []

for i, message in enumerate(test_messages, 1):
    print(f"\n{'='*60}")
    print(f"  TEST {i} of {len(test_messages)}")
    print(f"{'='*60}")

    result = run_agent(message)
    display_verdict(result)
    results.append(result)

    # Brief pause between tests
    time.sleep(2)


  ANTI-SCAM AGENT — PHASE 2 WITH TOOL CALLING
  Real evidence. Not guesses.

  TEST 1 of 3

  PHASE 2 AGENT — STARTING INVESTIGATION
Message preview: 
    From: hralert@wadialsagroup.com
    Subject: Interview Invitation
 
    Dear Candidate,
    We are pleased to invit...

  [Iteration 1] Calling LLM...

  → LLM requested: analyze_email_domain({'email': 'hralert@wadialsagroup.com'})

  🔍 [TOOL] Email domain analysis: hralert@wadialsagroup.com
     Domain       : wadialsagroup.com
     Free provider: False
     Suspicious prefix: True
     Has MX records: True
     Email risk   : MEDIUM

  → LLM requested: whois_lookup({'domain': 'wadialsagroup.com'})

  🔍 [TOOL] WHOIS lookup: wadialsagroup.com
     Domain age  : 273 days
     Risk level  : MEDIUM — domain less than 1 year old
     Country     : US
     Registrar   : GoDaddy.com, LLC

  → LLM requested: check_domain_mismatch({'body_domains': ['wadialsagroup.com'], 'sender_email': 'hralert@wadialsagroup.com'})

  🔍 [TOOL] Domain mismat

ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


     Domain age  : None days
     Risk level  : unknown
     Country     : not found
     Registrar   : not found

  → LLM requested: virustotal_scan({'url': 'http://emiratesnbd-secure-login.tk/verify'})

  🔍 [TOOL] VirusTotal scan: http://emiratesnbd-secure-login.tk/verify
     Scan submitted. Waiting for results...
     Malicious engines : 0/0
     Suspicious engines: 0/0
     Risk level        : LOW — not flagged by security engines
     (Waiting 15s for VirusTotal rate limit...)

  → LLM requested: scrape_website({'url': 'http://emiratesnbd-secure-login.tk/verify'})

  🔍 [TOOL] Scraping website: http://emiratesnbd-secure-login.tk/verify
     ⚠ Website unreachable — domain may not exist

  [Iteration 2] Calling LLM...

  ✅ Agent finished after 3 tool calls

────────────────────────────────────────────────────────────
  VERDICT : 🚨 SCAM DETECTED
  Confidence : [██████████] 100%
  Scam type  : Phishing
  Tools used : 3 tool calls made
──────────────────────────────────────────────────

In [ ]:

print("\n--- YOUR TURN: TEST YOUR OWN MESSAGE ---")

your_message = """

Online Jobs <info@talentcareerpath.com>
10:34 AM (4 hours ago)
to me

Dear Applicant,
Thank you for applying to ZYLX Group. We'd love to invite you for an initial interview — please use the link below to book a time that works for you:

Schedule Your Interview at http://www.zylx.online/
If the link doesn't work, reply with 2–3 preferred times and we'll sort it out.

Looking forward to speaking with you.

Best regards,
Abdulla Amani
Human Resources, ZYLX Group
"""

your_result = run_agent(your_message)
display_verdict(your_result)


--- YOUR TURN: TEST YOUR OWN MESSAGE ---

  PHASE 2 AGENT — STARTING INVESTIGATION
Message preview: 

Online Jobs <info@talentcareerpath.com>
10:34 AM (4 hours ago)
to me

Dear Applicant,
Thank you for applying to ZYLX G...

  [Iteration 1] Calling LLM...

  → LLM requested: analyze_email_domain({'email': 'info@talentcareerpath.com'})

  🔍 [TOOL] Email domain analysis: info@talentcareerpath.com
     Domain       : talentcareerpath.com
     Free provider: False
     Suspicious prefix: False
     Has MX records: True
     Email risk   : LOW

  → LLM requested: whois_lookup({'domain': 'talentcareerpath.com'})

  🔍 [TOOL] WHOIS lookup: talentcareerpath.com
     Domain age  : 171 days
     Risk level  : HIGH — domain less than 6 months old
     Country     : US
     Registrar   : GoDaddy.com, LLC

  → LLM requested: whois_lookup({'domain': 'zylx.online'})

  🔍 [TOOL] WHOIS lookup: zylx.online


ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


     Domain age  : 691 days
     Risk level  : LOW — domain is established
     Country     : not found
     Registrar   : Web Commerce Communications Limited dba WebNic.cc

  → LLM requested: virustotal_scan({'url': 'http://www.zylx.online/'})

  🔍 [TOOL] VirusTotal scan: http://www.zylx.online/
     Scan submitted. Waiting for results...
     Malicious engines : 0/0
     Suspicious engines: 0/0
     Risk level        : LOW — not flagged by security engines
     (Waiting 15s for VirusTotal rate limit...)

  → LLM requested: scrape_website({'url': 'http://www.zylx.online/'})

  🔍 [TOOL] Scraping website: http://www.zylx.online/
     Page title   : Home - ZYLX Online Jobs
     Flags found  : ['login_form', 'urgency_language']

  → LLM requested: check_domain_mismatch({'body_domains': ['zylx.online'], 'sender_email': 'info@talentcareerpath.com'})

  🔍 [TOOL] Domain mismatch check
     Sender domain : talentcareerpath.com
     Body domains  : ['zylx.online']
     Match found   : False
   